# LangGraph Series #3: Components and Your First Agent

This notebook accompanies the To Data & Beyond tutorial on LangGraph's core building blocks. You will create a ReAct-style research agent with shared state, model and action nodes, conditional routing, and Tavily web search.

## Prerequisites

The examples call OpenAI and Tavily, so you need `OPENAI_API_KEY` and `TAVILY_API_KEY`. In Google Colab, add both under **Secrets**. Locally, the setup cell prompts for any missing value without displaying it. API usage may incur provider charges.

In [ ]:
%pip install -qU langgraph langchain-openai langchain-community tavily-python

In [ ]:
import getpass
import os

def load_api_key(name: str) -> None:
    if os.getenv(name):
        return

    try:
        from google.colab import userdata
        value = userdata.get(name)
    except (ImportError, KeyError, TypeError):
        value = None

    os.environ[name] = value or getpass.getpass(f"Enter {name}: " )

load_api_key("OPENAI_API_KEY")
load_api_key("TAVILY_API_KEY")

## 1. Create the tool and shared state

The state contains one growing message list. The `operator.add` reducer tells LangGraph to append each node's new messages instead of replacing the history.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

tool = TavilySearchResults(max_results=4)
print(type(tool))
print(tool.name)

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

## 2. Build the agent graph

The graph has an LLM node and an action node. A conditional edge sends tool calls to the action node; otherwise the run ends. Tool results loop back to the model so it can answer or request another action.

In [ ]:
class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        self.tools = {tool.name: tool for tool in tools}
        self.model = model.bind_tools(tools)

        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END},
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()

    def exists_action(self, state: AgentState):
        result = state["messages"][-1]
        return len(result.tool_calls) > 0

    def call_openai(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system), *messages]
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []

        for tool_call in tool_calls:
            print(f"Calling: {tool_call}")
            tool_name = tool_call["name"]
            if tool_name not in self.tools:
                result = "Bad tool name. Select one of the available tools and retry."
            else:
                result = self.tools[tool_name].invoke(tool_call["args"])

            results.append(
                ToolMessage(
                    tool_call_id=tool_call["id"],
                    name=tool_name,
                    content=str(result),
                )
            )

        print("Back to the model!")
        return {"messages": results}

## 3. Initialize and visualize the agent

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information.
You may make multiple calls, either together or in sequence.
Only search when you are sure what information you need.
If you need information before asking a follow-up question, you may search for it first."""

model = ChatOpenAI(model="gpt-4o", temperature=0)
agent = Agent(model, [tool], system=prompt)

In [ ]:
from IPython.display import Image, display

display(Image(agent.graph.get_graph().draw_mermaid_png()))

## 4. Test a single search

Search results and the final answer will vary because weather, web content, and model behavior change over time.

In [ ]:
messages = [HumanMessage(content="What is the weather in Helsinki today?")]
result = agent.graph.invoke({"messages": messages})
print(result["messages"][-1].content)

## 5. Observe parallel tool calls

A model that supports parallel tool calling can request both city searches in the same turn.

In [ ]:
messages = [HumanMessage(content="What is the weather in Helsinki and Stockholm?")]
result = agent.graph.invoke({"messages": messages})
print(result["messages"][-1].content)

## 6. Test a multi-part research question

This query requires the agent to identify the 2022 World Cup winner and host before it can search for the two countries' GDP figures.

In [ ]:
query = (
    "Who won the last FIFA World Cup, where was it hosted, and what is the latest "
    "available GDP of each of those two countries? Answer every part."
)
messages = [HumanMessage(content=query)]
result = agent.graph.invoke({"messages": messages})
print(result["messages"][-1].content)

## What to try next

- Add another LangChain tool and confirm that the model routes to it.
- Stream graph events instead of waiting for the final state.
- Add a checkpointer so separate conversation threads retain their state.
- Insert a human approval step before selected tools run.